# trich xuat do thi ma nguon cho tap du lieu diversevul task 3.3

quy trinh thuc hien trich xuat hai bieu dien do thi cho 9077 ham
- graph a: do thi cay cu phap truu tuong ast
- graph b: do thi ngu nghia ket hop ast, luong dieu khien cfg va phu thuoc du lieu data_dep

cau hinh thuc thi tren moi truong cpu voi 4 tien trinh xu ly song song

### buoc 1. cai dat may ao java 21

In [ ]:
!apt-get update -qq && apt-get install -y -qq openjdk-21-jdk

### buoc 2. cai dat cong cu phan tich joern

In [ ]:
!curl -sSL https://github.com/joernio/joern/releases/download/v4.0.654/joern-install.sh | bash

### buoc 3. thiet lap bien moi truong thuc thi

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = "/root/bin:" + os.environ["PATH"]

### buoc 4. kiem tra phien ban cong cu

In [ ]:
!java -version
!joern --version

### buoc 5. dong bo ma nguon thuc thi

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b feature/3_3_graph-extract https://github.com/meo225/software-bug-detection-using-graphs.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyyaml

### buoc 6. xac thuc tep ma nguon diversevul

In [ ]:
from pathlib import Path

dataset_candidates = [
    Path("/kaggle/input/diversevul-raw/diversevul.jsonl"),
    Path("/kaggle/input/diversevul-dataset/diversevul.jsonl"),
    Path("/kaggle/input/diversevul/diversevul.jsonl"),
]

dataset_path = next((p for p in dataset_candidates if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("khong tim thay tep diversevul.jsonl tai duong dan dau vao")

print("duong dan tep ma nguon:", dataset_path)
print("dung luong tep:", round(dataset_path.stat().st_size / 1048576, 2), "mb")

### buoc 7. xac thuc tep manifest thi nghiem

In [ ]:
manifest_path = Path("data/splits/diversevul_experiment_manifest.csv")
if not manifest_path.is_file():
    raise FileNotFoundError(f"khong tim thay manifest tai {manifest_path}")

with open(manifest_path, "r", encoding="utf-8") as f:
    sample_count = sum(1 for _ in f) - 1

print("duong dan manifest:", manifest_path)
print("tong so mau thi nghiem:", sample_count)

### buoc 8. thuc thi trich xuat do thi theo phan doan

graph a la do thi cay cu phap truu tuong ast
graph b la do thi ngu nghia ket hop ast, luong dieu khien cfg va phu thuoc du lieu data_dep
bo comment mot dong tuong ung phan doan can chay

In [ ]:
# phan doan 1 tu mau 0 den 3000
!python scripts/extract_subset.py --dataset {dataset_path} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir /kaggle/working/output_batch_1 --batch 1 --workers 4

# phan doan 2 tu mau 3000 den 6000
# !python scripts/extract_subset.py --dataset {dataset_path} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir /kaggle/working/output_batch_2 --batch 2 --workers 4

# phan doan 3 tu mau 6000 den 9077
# !python scripts/extract_subset.py --dataset {dataset_path} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir /kaggle/working/output_batch_3 --batch 3 --workers 4

### buoc 9. kiem tra ket qua trich xuat

In [ ]:
import glob
from pathlib import Path

output_directories = sorted(glob.glob("/kaggle/working/output_batch_*"))
for out_dir in output_directories:
    p_dir = Path(out_dir)
    count_a = len(list((p_dir / "graph_a").glob("*.json"))) if (p_dir / "graph_a").is_dir() else 0
    count_b = len(list((p_dir / "graph_b").glob("*.json"))) if (p_dir / "graph_b").is_dir() else 0
    print("thu muc dau ra:", p_dir.name)
    print("so luong do thi cay cu phap truu tuong ast graph a:", count_a)
    print("so luong do thi ngu nghia ket hop graph b:", count_b)

### buoc 10. dong goi tao tac do thi

In [ ]:
import glob
import os

output_directories = sorted(glob.glob("/kaggle/working/output_batch_*"))
for out_dir in output_directories:
    batch_name = os.path.basename(out_dir)
    target_archive = f"/kaggle/working/{batch_name}.zip"
    print("thuc hien dong goi thu muc:", out_dir)
    !zip -q -r {target_archive} {out_dir}
    archive_size = os.path.getsize(target_archive) / 1048576
    print("tep luu tru hoan tat:", target_archive)
    print("dung luong tep:", round(archive_size, 2), "mb")